# Automatic Audio Description Generation for YouTube Shorts
##  Gemini 3.5 Flash Description Generation

**Author:** Spoorti Halappanavar  
**Student ID:** 6950243  
**Supervisor:** Dr Diptesh Kanojia  
**University:** University of Surrey  




### Overview
This notebook generates audio descriptions
for cooking YouTube Shorts using
Gemini 3.5 Flash via Google AI Studio API.

It covers:
1. **Setup** : Install libraries and configure API
2. **Prompts** : Zero-shot and five-shot prompts
3. **Generation** : Download videos and generate ADs


### Requirements
- Google AI Studio API key (Gemini 3.5 Flash)
- Input: cooking_shorts.csv from data collection notebook

### Output
- dataset with gemini_zero_shot column
- dataset with gemini_five_shot column

## Setup Section

In [ ]:
pip install google-genai yt-dlp -q
print(" Done!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 49.2 MB/s eta 0:00:00
✅ Done!


In [ ]:
from google import genai
from google.colab import userdata
import os
import time
import pandas as pd

API_KEY = userdata.get('YOUR_API_KEY_HERE')
client = genai.Client(api_key=API_KEY)
print(" Client ready!")

try:
    r = client.models.generate_content(
        model="gemini-3.5-flash",
        contents=["Say the number 1"]
    )
    print(f" Paid tier working! {r.text}")
except Exception as e:
    print(f" Error: {e}")

✅ Client ready!
✅ Paid tier working! 1


## Prompts Section

In [ ]:
# Prompts
zero_shot_prompt = """
You are a professional audio describer
creating descriptions for blind and
visually impaired viewers of food
related YouTube Shorts.

Your description will be read aloud
to someone who cannot see the screen
at all. Every word must be accurate
and confident.

Describe the video covering:
1. Who is in the video and where
2. What type of food video it is
3. The EXACT food being shown
   be specific — say "cheeseburger"
   not "sandwich"
4. Every key step in CORRECT order
5. Specific culinary techniques used
6. Colours textures and visual details
7. What each person is doing if
   multiple people are present
8. Any text visible on screen

Critical rules:
- NEVER use uncertain language like
  possibly appears to be seems might be
- Use EXACT food names not generic ones
- Describe steps in EXACT order
- Use correct culinary terminology
- Present tense only
- 3-5 sentences MAXIMUM
- Start description directly
  NO introduction sentences ever
- Never guess — only describe
  what you can clearly see
- Include ALL on screen text
  at the VERY END
- If no text visible skip entirely
"""

five_shot_prompt = """
You are a professional audio describer
creating descriptions for blind and
visually impaired viewers of cooking
YouTube Shorts.

Your description will be read aloud to
someone who cannot see the screen. Every
word must be accurate, strictly grounded
in literal on-screen reality, and use
correct culinary terminology.

Study these 5 examples carefully. Notice
how they remain entirely faithful to what
is visibly proven on screen without ever
assuming, guessing, or fabricating
out-of-frame context:

EXAMPLE 1 — Burger assembly outdoors:
"Two men sit at a wooden picnic table
outdoors. One man spreads a creamy white
sauce onto the bottom half of a freshly
baked poppy seed bun, then adds ketchup
and yellow mustard. He layers sliced green
pickles over the sauces before placing a
thick beef patty smothered in melted white
cheese on top. Two strips of crispy bacon
complete the filling before the seeded top
bun is placed on the towering cheeseburger.
Text reads: Silently Cooking"

EXAMPLE 2 — Soup with two cooks:
"Two men work side by side at a wooden
counter in a bright kitchen around a large
silver stockpot. One man pours bright
yellow corn kernels and chopped green beans
directly into a bubbling tomato broth while
the other grates a fresh garlic clove over
a cutting board. The rich red stew thickens
as it cooks before being ladled into wide
serving bowls and garnished with finely
shredded white cheese. The word minestrone
appears briefly in the centre of the screen."

EXAMPLE 3 — Pastry with technique:
"A baker works on a flour-dusted wooden
board rolling out a block of dough with a
heavy rolling pin. The dough is folded
neatly into thirds to create distinct
layers before being rolled into a long
uniform sheet. A knife cuts the sheet into
clean triangles which are rolled by hand
from the wide base to the pointed tip to
form classic crescent shapes. The croissants
bake until golden brown with a visible
flaky sheen on their layered crusts."

EXAMPLE 4 — Solo cooking multiple steps:
"A woman in a white apron stands at a
marble counter and halves two ripe avocados
before scooping the green flesh into a
white bowl. She mashes the avocado with a
fork until chunky then folds in finely
diced red onion, chopped tomatoes and a
generous squeeze of fresh lime juice. The
finished guacamole is spooned into a
serving bowl alongside a stack of golden
tortilla chips. Text reads: 5 Minute
Guacamole"

EXAMPLE 5 — Stir fry with fast cuts:
"A chef heats a generous pour of vegetable
oil in a large carbon steel wok over high
heat until it shimmers. Day old white rice
is added and tossed continuously with a
wooden spatula until each grain separates
and turns golden. Two beaten eggs are
pushed to the centre of the wok and
scrambled before being folded through the
rice with soy sauce and sliced spring
onions. The fried rice is plated in a
white bowl and finished with a drizzle
of sesame oil."


Now describe this cooking video using
exactly the same style and quality.

Critical rules:
- STRICT LITERAL GROUNDING: Only describe
  what you can explicitly verify with your
  eyes in this exact clip. Never guess,
  assume, or infer hidden actions. If an
  ingredient is not visibly on screen
  omit it entirely
- Start directly — NO introduction
- Present tense throughout
- 3-5 sentences only
- Use SPECIFIC food names
  e.g. "cheeseburger" NOT "sandwich"
- NEVER say possibly appears to be
  seems might be — be confident
  but factual
- Describe steps in EXACT order
- Use culinary terms correctly
- If two people are cooking describe
  BOTH actions at the same time
- Include literal colours and textures
- Include any on screen text at the end
"""

print(" Both prompts ready!")
print(" Same prompts as Qwen script!")

✅ Both prompts ready!
✅ Same prompts as Qwen script!


## Generation Pipeline

In [ ]:
# Functions

import yt_dlp

def download_video(url,
                   path="/content/temp_video.mp4"):
    if os.path.exists(path):
        os.remove(path)
    for client_type in ['android', 'ios', 'web']:
        ydl_opts = {
            'outtmpl': path,
            'format': 'worst[ext=mp4]/best',
            'quiet': True,
            'no_warnings': True,
            'extractor_args': {
                'youtube': {
                    'player_client': [client_type]
                }
            },
        }
        try:
            with yt_dlp.YoutubeDL(ydl_opts) as ydl:
                ydl.download([url])
            if os.path.exists(path):
                size = os.path.getsize(path)
                print(f"   {size/1024:.1f}KB")
                return path
        except:
            continue
    print("   Download failed!")
    return None


def generate_with_retry(contents,
                        max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-3.5-flash",
                contents=contents
            )
            return response
        except Exception as e:
            error = str(e)
            if "503" in error:
                wait = (attempt + 1) * 10
                print(f"   Server busy! "
                      f"Waiting {wait}s...")
                time.sleep(wait)
            elif "429" in error:
                print(f"   Rate limit! "
                      f"Waiting 30s...")
                time.sleep(30)
            else:
                print(f"   Error: {e}")
                return None
    print("   Failed after retries!")
    return None

print(" Functions ready!")

✅ Functions ready!


In [ ]:
import pandas as pd
df = pd.read_csv("/content/cooking_shorts.csv")

done_check = (df['annotation_status'] == 'AI_Done').sum()
pending_check = (df['annotation_status'] == 'Pending').sum()
print(f" Fresh load! Done: {done_check} | Pending: {pending_check}")


MAX_VIDEOS = 224
SAVE_EVERY = 10

processed = 0
errors_count = 0
start_time = time.time()

print(f" Starting! Target: {MAX_VIDEOS}")
print(f" Paid tier — no rate limits!")
print(f" Est: ~{MAX_VIDEOS*22/60:.0f} mins")
print(f" Est cost: ~£{MAX_VIDEOS*0.013:.2f}")
print(f"{'='*50}")

for index, row in df.iterrows():

    if processed >= MAX_VIDEOS:
        print(f"\n Done {MAX_VIDEOS} today!")
        break

    if row['annotation_status'] in [
        'AI_Done', 'Error'
    ]:
        continue

    url = str(row['shorts_url'])
    title = str(row['title'])[:40]

    if processed > 0:
        elapsed = time.time() - start_time
        avg = elapsed / processed
        eta = (MAX_VIDEOS-processed)*avg/60
        cost = processed * 0.013
        print(f"\n[{processed+1}/{MAX_VIDEOS}]"
              f" ETA:{eta:.0f}m"
              f" £{cost:.2f} | {title}")
    else:
        print(f"\n[1/{MAX_VIDEOS}] {title}")

    try:
        # Download
        video_path = download_video(url)
        if not video_path:
            df.at[index,
                  'annotation_status'] = 'Error'
            errors_count += 1
            continue

        # Upload
        print("  Uploading...")
        uploaded = client.files.upload(
            file=video_path
        )
        print("   Uploaded!")
        time.sleep(3)

        # Zero Shot
        print("  Zero shot...")
        r1 = generate_with_retry(
            [zero_shot_prompt, uploaded]
        )
        zero_result = r1.text if r1 else "Failed"
        print(f"   {zero_result[:55]}...")

        time.sleep(3)

        # Five Shot
        print("  Five shot...")
        r2 = generate_with_retry(
            [five_shot_prompt, uploaded]
        )
        five_result = r2.text if r2 else "Failed"
        print(f"   {five_result[:55]}...")

        # Save to dataframe
        df.at[index, 'gemini_zero_shot'] = \
            zero_result
        df.at[index, 'gemini_five_shot'] = \
            five_result
        df.at[index, 'annotation_status'] = \
            'AI_Done'

        # Clean up
        try:
            client.files.delete(
                name=uploaded.name
            )
            os.remove(video_path)
        except:
            pass

        processed += 1

        # Save every 10 videos
        if processed % SAVE_EVERY == 0:
            df.to_csv(
                "/content/cooking_shorts.csv",
                index=False
            )
            elapsed_m = \
                (time.time()-start_time)/60
            cost = processed * 0.013
            print(f"\n  💾 Saved! "
                  f"{processed} done | "
                  f"{elapsed_m:.1f}m | "
                  f"£{cost:.2f}")

        # Small gap
        time.sleep(2)

    except Exception as e:
        print(f"   Error: {e}")
        df.at[index,
              'annotation_status'] = 'Error'
        errors_count += 1
        if os.path.exists(
            "/content/temp_video.mp4"
        ):
            os.remove(
                "/content/temp_video.mp4"
            )
        continue

df.to_csv(
    "/content/cooking_shorts.csv",
    index=False
)

total_time = (time.time()-start_time)/60
total_cost = processed * 0.013
done_total = (
    df['annotation_status'] == 'AI_Done'
).sum()

print(f"\n{'='*50}")
print(f" SESSION COMPLETE!")
print(f"   Processed:  {processed}")
print(f"   Errors:     {errors_count}")
print(f"   Total done: {done_total}/{len(df)}")
print(f"   Time:       {total_time:.1f} mins")
print(f"   Cost:       ~£{total_cost:.2f}")
print(f"{'='*50}")

from google.colab import files
files.download("/content/cooking_shorts.csv")

✅ Fresh load! Done: 243 | Pending: 224
🚀 Starting! Target: 224
💳 Paid tier — no rate limits!
⏱️ Est: ~82 mins
💰 Est cost: ~£2.91

[1/224] Not So Silently Cooking Short - Burger H
  ✅ 3776.4KB
  Uploading...
  ✅ Uploaded!
  Zero shot...
  ⚠️ Server busy! Waiting 10s...
  ✅ Two men sit at an outdoor wooden table, preparing to ea...
  Five shot...
  ✅ Two men sit at a wooden picnic table outdoors, holding ...

[2/224] ETA:294m £0.01 | Slow down and build a burger
  ✅ 3510.8KB
  Uploading...
  ✅ Uploaded!
  Zero shot...
  ✅ A chef outdoors assembles a double bacon cheeseburger o...
  Five shot...
  ✅ A hand spreads a creamy white sauce over the bottom hal...

[3/224] ETA:212m £0.03 | Silently Cooking Short - Minestrone 1 #m
  ✅ 4496.2KB
  Uploading...
  ✅ Uploaded!
  Zero shot...
  ✅ A man in a kitchen prepares a bean stew by soaking red ...
  Five shot...
  ✅ A cook pours dried red kidney beans into a tall plastic...

[4/224] ETA:187m £0.04 | Silently Cooking - Spanish Tortilla
  ✅ 10584.

ERROR: [youtube] -DyXkuYAP1k: This video contains content from Wixen Music Publishing Inc.. It is not available in your country. Learn more
ERROR: [youtube] -DyXkuYAP1k: Video unavailable. This video contains content from Wixen Music Publishing Inc.. It is not available in your country. Learn more
ERROR: [youtube] -DyXkuYAP1k: Video unavailable. This video contains content from Wixen Music Publishing Inc.. It is not available in your country. Learn more


  ❌ Download failed!

[6/224] ETA:361m £0.07 | I brought in a coffee specialist @FlairE
  ✅ 24.3KB
  Uploading...
  ✅ Uploaded!
  Zero shot...
  ❌ Error: 400 FAILED_PRECONDITION. {'error': {'code': 400, 'message': 'The File g5v3sgjggw30 is not in an ACTIVE state and usage is not allowed.', 'status': 'FAILED_PRECONDITION'}}
  ✅ Failed...
  Five shot...
  ❌ Error: 400 FAILED_PRECONDITION. {'error': {'code': 400, 'message': 'The File g5v3sgjggw30 is not in an ACTIVE state and usage is not allowed.', 'status': 'FAILED_PRECONDITION'}}
  ✅ Failed...

[7/224] ETA:306m £0.08 | The infamous paunch burger from Parks & 
  ✅ 4322.7KB
  Uploading...
  ✅ Uploaded!
  Zero shot...
  ✅ A male chef in a kitchen prepares a multi-layered chees...
  Five shot...
  ✅ A cook toasts bun halves on a square cast-iron skillet,...

[8/224] ETA:339m £0.09 | Luca's iconic trenette al pesto pasta ??
  ✅ 4384.0KB
  Uploading...
  ✅ Uploaded!
  Zero shot...
  ✅ In a split-screen format contrasting clips from the ani..

ERROR: [youtube] npZEwO0wh6g: The uploader has not made this video available in your country
ERROR: [youtube] npZEwO0wh6g: The uploader has not made this video available in your country
This video is available in United Kingdom.
You might want to use a VPN or a proxy server (with --proxy) to workaround.
ERROR: [youtube] npZEwO0wh6g: The uploader has not made this video available in your country
This video is available in United Kingdom.
You might want to use a VPN or a proxy server (with --proxy) to workaround.


  ❌ Download failed!

[68/224] ETA:151m £0.87 | Gordon Ramsay cooks with ??@MrBeast in M
  ✅ 6520.0KB
  Uploading...
  ✅ Uploaded!
  Zero shot...
  ✅ In a restaurant dining room, Jimmy wears a black suit a...
  Five shot...
  ✅ A chef in a white uniform and a man in a black suit sta...

[69/224] ETA:151m £0.88 | Gordon Ramsay Turns a Full English Break
  ✅ 6536.4KB
  Uploading...
  ✅ Uploaded!
  Zero shot...
  ✅ Chef Gordon Ramsay prepares a hearty breakfast sandwich...
  Five shot...
  ✅ A chef fries sausages and strips of pork bacon in a hot...

[70/224] ETA:149m £0.90 | Gordon RamsayÕs Beef Lettuce CupsÉ.perfe
  ✅ 5951.5KB
  Uploading...
  ✅ Uploaded!
  Zero shot...
  ✅ Chef Gordon Ramsay stands in a modern kitchen preparing...
  Five shot...
  ✅ A chef in a black t-shirt seasons raw red ground beef o...

  💾 Saved! 70 done | 67.1m | £0.91

[71/224] ETA:148m £0.91 | Using Instant Ramen Noodles to make a Ne
  ✅ 4345.8KB
  Uploading...
  ✅ Uploaded!
  Zero shot...
  ✅ In a professiona

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Save and Download Results
import os
from google.colab import files

# Save updated dataset
df.to_csv(
    "/content/dataset_with_descriptions.xlsx",
    index=False
)
print(f" Saved!")
print(f"Gemini Zero: {(df['gemini_zero_shot'] != '').sum()}")
print(f"Gemini Five: {(df['gemini_five_shot'] != '').sum()}")

# Download
files.download(
    "/content/dataset_with_descriptions.xlsx"
)